<a href="https://colab.research.google.com/github/1kaiser/Media-Segment-Depth-MLP/blob/main/pixel_perfect_depth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/gangweix/pixel-perfect-depth
%cd pixel-perfect-depth
!uv pip install -r requirements.txt

In [ ]:
!rm -rf checkpoints
!mkdir -p checkpoints
!wget https://huggingface.co/depth-anything/Depth-Anything-V2-Large/resolve/main/depth_anything_v2_vitl.pth -O checkpoints/depth_anything_v2_vitl.pth
!wget https://huggingface.co/gangweix/Pixel-Perfect-Depth/resolve/main/ppd.pth -O checkpoints/ppd.pth
!wget https://huggingface.co/Ruicheng/moge-2-vitl-normal/resolve/main/model.pt -O checkpoints/moge2.pt


In [ ]:
!python run.py --save_npy

In [ ]:
# Run this in a Colab cell
!ls -lh checkpoints/moge2.pt

In [ ]:
!python run_point_cloud.py --save_pcd

In [ ]:
!apt install tree
!tree

In [ ]:
import numpy as np
!uv pip install --upgrade trimesh
import trimesh
import cv2

# --- 1. Load Data ---
depth_file = 'depth_npy/0001.npy'
image_file = 'assets/examples/0001.jpg'

depth = np.load(depth_file)
image = cv2.imread(image_file)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# --- 2. Unproject Depth to 3D Points ---
h, w = depth.shape
x, y = np.arange(w), np.arange(h)
xx, yy = np.meshgrid(x, y)

fx = fy = w
cx, cy = w / 2, h / 2

x_cam = (xx - cx) * depth / fx
y_cam = (yy - cy) * depth / fy
z_cam = depth

points = np.stack([x_cam, y_cam, z_cam], axis=-1).reshape(-1, 3)
colors = image.reshape(-1, 3)

# --- 3. Downsample and Create Point Cloud ---
num_points = points.shape[0]
if num_points > 300000:
    step = num_points // 300000
    points = points[::step]
    colors = colors[::step]

cloud = trimesh.points.PointCloud(vertices=points, colors=colors)

# --- 4. ENHANCEMENT: Center the Geometry ---
# We calculate the center of the point cloud and move the whole cloud
# so that its new center is at the world origin [0, 0, 0].
# This makes the orbital camera rotate around the object's actual center.
center = cloud.bounds.mean(axis=0)
cloud.apply_translation(-center)

# --- 5. Create a Scene and Set a Better Camera Angle ---
scene = trimesh.Scene(cloud)

# Define a 45-degree rotation around the X-axis and Z-axis for a nice 3/4 view.
# This is more intuitive than setting the matrix manually.
angle_x = np.deg2rad(45)
angle_z = np.deg2rad(-45)
rot_x = trimesh.transformations.rotation_matrix(angle=angle_x, direction=[1, 0, 0])
rot_z = trimesh.transformations.rotation_matrix(angle=angle_z, direction=[0, 0, 1])

# Combine the rotations and set it as the camera transform.
# The scene will now start from this viewpoint.
scene.camera.transform = rot_z @ rot_x

# --- 6. Display the Interactive Viewer ---
# Using scene.show() which works well in notebook environments.
# The viewer itself has the orbital controls built-in (click and drag).
scene.show()

In [ ]:
from IPython.display import display, HTML
import json
import os
import base64

# --- 1. Load the PLY file directly from its path IN BINARY MODE ---
file_path = '/content/pixel-perfect-depth/depth_pcd/0001.ply'

if not os.path.exists(file_path):
    print(f"ERROR: The file was not found at the specified path: {file_path}")
    print("Please make sure you have run the point cloud generation script first.")
else:
    # Read the file in binary mode ('rb')
    with open(file_path, 'rb') as f:
        binary_content = f.read()

    # Encode the binary content into a Base64 string.
    # This is a safe way to embed binary data in a text-based format like HTML/JS.
    base64_content = base64.b64encode(binary_content).decode('ascii')

    # Use json.dumps to safely escape the Base64 string for JavaScript
    ply_data_for_js = json.dumps(base64_content)
    print(f"Successfully loaded and encoded '{os.path.basename(file_path)}'. Generating the viewer...")

    # --- 2. Generate and Display the Interactive Viewer ---
    # The JavaScript is updated to handle the Base64 data.
    html_template = f"""
    <!DOCTYPE html>
    <html>
    <head>
    <title>Colab 3D Measurement Tool</title>
    <style>
      body {{ font-family: sans-serif; }}
      #viewer {{ width: 100%; height: 600px; border: 1px solid #ccc; }}
      #info {{ margin-top: 10px; font-size: 16px; font-weight: bold; }}
    </style>

    <script type="module" src="https://ajax.googleapis.com/ajax/libs/model-viewer/3.5.0/model-viewer.min.js"></script>
    <script src="https://cdn.jsdelivr.net/npm/three@0.138.0/build/three.min.js"></script>
    <script src="https://cdn.jsdelivr.net/npm/three@0.138.0/examples/js/loaders/PLYLoader.js"></script>
    </head>
    <body>

      <h1>Interactive 3D Point Cloud Measurement</h1>
      <p>Click any two points on the model to measure the distance between them.</p>
      <model-viewer id="viewer" camera-controls ar touch-action="pan-y" alt="3D point cloud viewer"></model-viewer>
      <div id="info">Measurement results will appear here.</div>

      <script>
        const modelViewer = document.getElementById('viewer');
        const infoDiv = document.getElementById('info');

        let firstPoint = null;
        let scene = null;

        // The PLY data is now a Base64 encoded string.
        const base64Data = {ply_data_for_js};

        // NEW: Helper function to decode Base64 into an ArrayBuffer
        function base64ToArrayBuffer(base64) {{
            const binaryString = window.atob(base64);
            const len = binaryString.length;
            const bytes = new Uint8Array(len);
            for (let i = 0; i < len; i++) {{
                bytes[i] = binaryString.charCodeAt(i);
            }}
            return bytes.buffer;
        }}

        function loadPointCloud() {{
          // Decode the Base64 data back into binary
          const arrayBuffer = base64ToArrayBuffer(base64Data);

          const loader = new THREE.PLYLoader();
          // The loader can parse an ArrayBuffer directly, correctly handling binary PLY files.
          const geometry = loader.parse(arrayBuffer);

          const material = new THREE.PointsMaterial({{ vertexColors: true, size: 0.01 }});
          const points = new THREE.Points(geometry, material);

          modelViewer.addEventListener('load', () => {{
            scene = modelViewer.shadowRoot.querySelector('canvas').__private_three_scene;
            scene.add(points);
          }}, {{ once: true }});

          modelViewer.src = 'data:text/plain;base64,';
          infoDiv.textContent = 'Model loaded. Ready to measure.';
        }}

        modelViewer.addEventListener('click', (event) => {{
          if (!scene) return;
          const hit = modelViewer.positionAndNormalFromPoint(event.clientX, event.clientY);
          if (hit) {{
            const clickedVec3 = new THREE.Vector3(hit.position.x, hit.position.y, hit.position.z);
            if (!firstPoint) {{
              firstPoint = clickedVec3;
              infoDiv.textContent = `First point selected. Click a second point.`;
            }} else {{
              const distance = firstPoint.distanceTo(clickedVec3);
              infoDiv.textContent = `Distance: ${{distance.toFixed(4)}} meters (${{(distance * 100).toFixed(2)}} cm).`;
              firstPoint = null;
            }}
          }}
        }});

        loadPointCloud();
      </script>

    </body>
    </html>
    """

    display(HTML(html_template))